In [1]:
import os
import glob as glob
import matplotlib.pyplot as plt
import cv2
import requests
import random
import numpy as np

SEED = 42
np.random.seed(SEED)

In [22]:
TRAIN = True

EPOCHS = 100

In [16]:
class_names = ["UAV"]
color = (0,0,255)

In [17]:
# convert bounding boxes into YOLO format; xmin, ymin, xmax, ymax
def yolo2bbox(bboxes):
    xmin, ymin = bboxes[0]-bboxes[2]/2, bboxes[1]-bboxes[3]/2
    xmax, ymax = bboxes[0]+bboxes[2]/2, bboxes[1]+bboxes[3]/2
    return xmin, ymin, xmax, ymax


In [18]:
def plot_box(image, bboxes, labels):
    #need image height and width to denormalize the bounding box coordinates
    h, w, _= image.shape
    for box_num, box in enumerate(bboxes):
        x1,y1,x2,y2 = yolo2bbox(box)
        # denormalize coordinates
        xmin = int(x1*w)
        ymin = int(y1*h)
        xmax = int(x2*w)
        ymax = int(y2*h)
        width = xmax - xmin
        height = ymax - ymin

        class_name = class_names[int(labels[box_num])]

        cv2.rectangle(
            image,
            (xmin, ymin), (xmax, ymax),
            color = color,
            thickness = 2
        )

        font_scale = min(1,max(3,int(w/500)))
        font_thickness = min(2,max(10,int(w/50)))

        p1,p2 = (int(xmin), int(ymin)), (int(xmax), int(ymax))
        #text width and height
        tw, th = cv2.getTextSize(
            class_name,
            0, fontScale=font_scale, thickness=font_thickness
        )[0]
        p2 = p1[0]+tw, p1[1]+ -th -10
        cv2.rectangle(
            image,
            p1,p2,
            color=color,
            thickness=1
        )
        cv2.putText(
            image, 
            class_name,
            (xmin+1, ymin-10),
            cv2.FONT_HERSHEY_SIMPLEX,
            font_scale,
            (255,255,255),
            font_thickness
        )
    return image

In [19]:
# function to plot images with bounding boxes
def plot(image_paths, label_paths, num_samples):
    all_training_images = glob.glob(image_paths)
    all_training_labels = glob.glob(label_paths)
    all_training_images.sort()
    all_training_labels.sort()

    num_images = len(all_training_images)

    plt.figure(figsize=(15,12))
    for i in range(num_samples):
        j = random.randint(0, num_images-1)
        image = cv2.imread(all_training_images[j])
        with open(all_training_labels[j], 'r') as f:
            bboxes = []
            labels = []
            label_lines = f.readlines()
            for label_line in label_lines:
                label = label_line[0]
                bbox_string = label_line[2:]
                x_c, y_c,w,h = bbox_string.split(' ')
                x_c = float(x_c)
                y_c = float(y_c)
                w = float(w)
                h = float(h)
                bboxes.append([x_c, y_c, w, h])
                labels.append(label)
        result_image = plot_box(image, bboxes, labels)
        plt.subplot(2, 2, i+1)
        plt.imshow(result_image[:, :, ::-1])
        plt.axis('off')
    plt.subplots_adjust(wspace=0)
    plt.tight_layout()
    plt.show()




In [20]:
# visualize a few training images
plot(
    image_paths='../prepared_dataset/images/train/*__visible__*',
    label_paths='../prepared_dataset/labels/train/*__visible__*',
    num_samples=4,
)

ValueError: empty range in randrange(0, 0)

<Figure size 1500x1200 with 0 Axes>

### Helper functions

used for logging of results in the notebokk while training the models

In [21]:
def set_res_dir():
    # directory to store results
    res_dir_count = len(glob.glob('runs/train/*'))
    print(f"Current number of result directories: {res_dir_count}")
    if TRAIN:
        RES_DIR = f"results_{res_dir_count+1}"
        print(RES_DIR)
    else:
        RES_DIR = f"results_{res_dir_count}"
    return RES_DIR

tensorboard, loads tensorboard and monitors

In [9]:
def monitor_tensorboard():
    %load_ext tensorboard
    %tensorboard --logdir runs/train

# tba


## Clone YOLOV5 Repo

In [10]:
import os
os.environ["PATH"] += r";C:\Program Files\Git\mingw64\bin"


In [11]:
if not os.path.exists('yolov5'):
    !git clone https://github.com/ultralytics/yolov5.git

In [12]:
%cd yolov5/
!pwd

c:\Users\Gur Levy\Desktop\UVA\MASTER\THESIS\thesis-1\yolo\yolov5


'pwd' is not recognized as an internal or external command,
operable program or batch file.


In [13]:
!pip install -r requirements.txt

## Training

In [ ]:
RES_DIR = set_res_dir()
if TRAIN:
    !python train.py --data ../../anti_uav.yaml --weights yolov5m.pt \
    --img 640 --epochs {EPOCHS} --batch-size 16 --name {RES_DIR}

Current number of result directories: 2
results_3
^C


train: weights=yolov5m.pt, cfg=, data=../../anti_uav.yaml, hyp=data\hyps\hyp.scratch-low.yaml, epochs=25, batch_size=16, imgsz=640, rect=False, resume=False, nosave=False, noval=False, noautoanchor=False, noplots=False, evolve=None, evolve_population=data\hyps, resume_evolve=None, bucket=, cache=None, image_weights=False, device=, multi_scale=False, single_cls=False, optimizer=SGD, sync_bn=False, workers=8, project=runs\train, name=results_3, exist_ok=False, quad=False, cos_lr=False, label_smoothing=0.0, patience=100, freeze=[0], save_period=-1, seed=0, local_rank=-1, entity=None, upload_dataset=False, bbox_interval=-1, artifact_alias=latest, ndjson_console=False, ndjson_file=False
github: up to date with https://github.com/ultralytics/yolov5 
fatal: cannot change to 'C:\Users\Gur': Invalid argument
YOLOv5  2026-4-14 Python-3.13.2 torch-2.11.0+cpu CPU

hyperparameters: lr0=0.01, lrf=0.01, momentum=0.937, weight_decay=0.0005, warmup_epochs=3.0, warmup_momentum=0.8, warmup_bias_lr=0.1, b